# Lecture 7 (17 Oct 2026) — Twelve small tasks with pandas and an LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kosavina/hse-python-2026/blob/main/Lecture_7.ipynb)

*Student notebook. The model writes the code. You decide what it must compute, you check it, and you decide when it is right.*

Twelve short tasks. Each uses **one idea from Lecture 6** on a table small enough to check by eye, or asks one question about SBER 2025. Every task follows the same five steps as Lecture 5:

```
 1 DISCUSS  ──►  2 PROMPT  ──►  3 VERIFY  ──►  4 FEATURE  ──►  5 VERIFY AGAIN
  look at the     the model       check cell      the model        check cell
  data first                         │                                 │
                                     └───── failed? FIX with the model ┘
```

1. **Discuss.** Look at the data and answer one or two questions: usually *how many rows will come out?* or *what is this number by hand?* Write the answers in `HAND_n`.
2. **Prompt.** Write the prompt in the box, send it to the model, and paste the code into the empty cell. The spec lists the names the check cell expects; put them in your prompt.
3. **Verify.** Run the check cell (don't edit it). Then answer the reading questions.
4. **Feature.** Paste your working code back into the chat and ask for the change. Ask the model to *change* the code rather than start again.
5. **Verify again.** The second check cell.

**If a check fails:** paste the failed `assert` line and its message into the chat, ask the model to **explain which line causes it first**, and only then to fix it. Rerun **all** the check cells of the task, not just the one that failed.

**When you ask a model for pandas code** (from the end of Lecture 6):

1. **Describe the table**: the column names, what the index is, the number of rows. For the small tables, paste the "Data for this task" cell into the chat.
2. **Ask it to print `.shape`** after every step that can change the row count.
3. **For every merge, name the key, `how=` and `validate=`.**
4. **Say what to do with the first `NaN`** (`pct_change`, `shift`).
5. **Ask for `.loc` / `.iloc`**, never `df[...][...] = ...`.
6. **Give the exact variable names** from the spec, so the check cell finds them.

Three methods appear here for the first time: `sort_values` (Tasks 4 and 10), `isin` (Task 8) and `pivot_table` (Task 12). The spec names them; read the model's code and find them.

AI level: Full AI (AIAS 4). Fill in the AI log at the end.

## Setup — pandas and the SBER file

In [ ]:
# Setup. Run this first.
import os, urllib.request
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/kosavina/hse-python-2026/main/data/SBER_2025_daily.csv"
DATA_FILE = "SBER_2025_daily.csv"
for candidate in (DATA_FILE, os.path.join("data", DATA_FILE), os.path.join("..", "data", DATA_FILE)):
    if os.path.exists(candidate):
        DATA_FILE = candidate
        break
else:
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

print("pandas", pd.__version__, "| data file:", DATA_FILE)

---
# A · Build, load, clean
Make a table from scratch, read one from a file, and find the gaps in it.

---
## Task 1 — My portfolio as a table

**The story.** You hold five Russian stocks. The broker app shows a list; you want a **table**: one row per ticker, and the value of each position.

In [ ]:
# Data for Task 1: what you hold, and the price of one share (prices rounded for the exercise)
data = {
    "ticker": ["SBER", "GAZP", "LKOH", "YDEX", "MGNT"],
    "shares": [100, 300, 5, 10, 4],
    "price":  [300.0, 120.0, 6000.0, 4000.0, 3500.0],
}

### 1.1 Discuss
1. By hand: the value of each position (shares × price), and the total. Which position is the largest?
2. Each row is a ticker. What should the **index** be? What is the shape of the table with a `value` column?

In [ ]:
HAND_1 = "..."   # your hand answers

**The spec:** `portfolio` — a DataFrame built from `data`, with the tickers as the index (named `ticker`) and the columns `shares`, `price` and `value` = shares × price.

### 1.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 1.3 Verify *(don't edit)*

In [ ]:
assert len(portfolio) == 5, f"one row per ticker, got {len(portfolio)}"
assert portfolio.index.name == "ticker", "the tickers must be the index, not a column"
assert {"shares", "price", "value"} <= set(portfolio.columns), list(portfolio.columns)
assert portfolio.loc["LKOH", "value"] == 30_000, "LKOH: 5 shares x 6000"
assert portfolio["value"].sum() == 150_000, portfolio["value"].sum()
print("OK: the portfolio is worth", f"{portfolio['value'].sum():,.0f}", "RUB")

**Read the code.**
1. What would the shape be without `set_index`? What would `portfolio.loc["LKOH"]` do then?
2. The line that makes `value`: is there a loop over the rows? How does pandas know which price goes with which number of shares?

### 1.4 Feature — weights
**The spec:** add a column `weight` = value / total value, and make `top` the **ticker** with the largest weight.

*Before you run:* what are the weights of SBER and MGNT?

### 1.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert "weight" in portfolio.columns
assert np.isclose(portfolio["weight"].sum(), 1), "the weights must add up to 1"
assert np.isclose(portfolio.loc["SBER", "weight"], 0.2), "SBER: 30 000 / 150 000"
assert top == "YDEX", f"`top` must be a ticker (a label), got {top!r}"
print("OK: weights", portfolio["weight"].round(3).to_dict())

---
## Task 2 — Load SBER properly

**The story.** In Lecture 6 you saw the SBER file read twice: once with the dates as plain text, once as real dates. Now ask the model to load it the right way, in one line, and add two useful columns.

The file name is in `DATA_FILE` (from the setup cell). Columns: `TRADEDATE, OPEN, HIGH, LOW, CLOSE, VOLUME` (shares), `VALUE` (rubles).

### 2.1 Discuss
1. If you call `pd.read_csv(DATA_FILE)` with no other arguments, what is the dtype of `TRADEDATE`? Can you then write `sber.loc["2025-11"]`?
2. With the dates as the index: how many rows and how many columns?

In [ ]:
HAND_2 = "..."   # your hand answers

**The spec:** `sber` — a DataFrame read from `DATA_FILE`, with the dates as a `DatetimeIndex` named `TRADEDATE`, and the six price/volume columns.

### 2.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 2.3 Verify *(don't edit)*

In [ ]:
assert len(sber) == 254, f"254 trading days, got {len(sber)}"
assert {"OPEN", "HIGH", "LOW", "CLOSE", "VOLUME", "VALUE"} <= set(sber.columns), list(sber.columns)
assert "TRADEDATE" not in sber.columns and sber.index.name == "TRADEDATE", "the dates must be the index"
assert isinstance(sber.index, pd.DatetimeIndex), f"the index must hold dates, not text: {sber.index.dtype}"
assert sber["CLOSE"].dtype == float and sber.loc["2025-01-03", "CLOSE"] == 272.25
assert len(sber.loc["2025-11"]) == 20, "November 2025 has 20 trading days"
print("OK:", sber.index[0].date(), "...", sber.index[-1].date(), "|", len(sber), "days")

**Read the code.**
1. Which argument turns the text into dates? Which one makes them the index?
2. Why is `VOLUME` an integer column and `VALUE` a float one?

### 2.4 Feature — two new columns
**The spec:** add `turnover_bn` = VALUE in billion rubles, and `range_pct` = (HIGH − LOW) / OPEN, the day's range as a share of the open. `widest_day` = the date with the largest `range_pct`.

*Before you run:* which day of 2025 do you expect? (Lecture 6: the best day of the year.)

### 2.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert len(sber) == 254, "new columns, the same 254 rows"
assert np.isclose(sber.loc["2025-01-03", "turnover_bn"], 11.8535659849), "VALUE is in rubles: divide by 1e9"
assert (sber["range_pct"] >= 0).all(), "HIGH >= LOW, so the range can't be negative"
assert np.isclose(sber["range_pct"].max(), 0.0784913, atol=1e-6), "(HIGH - LOW) / OPEN, as a share, not in %"
assert pd.Timestamp(widest_day) == pd.Timestamp("2025-02-12"), widest_day
print("OK: widest day", pd.Timestamp(widest_day).date(), f"range {sber['range_pct'].max():.2%}")

**Stuck on Task 2?** Run the next cell, so that the SBER tasks (4, 5, 6, 11) still work.

In [ ]:
# Run this cell ONLY if your Task 2 checks failed: it builds `sber`, so Tasks 4, 5, 6 and 11 still work.
sber = pd.read_csv(DATA_FILE, parse_dates=["TRADEDATE"], index_col="TRADEDATE")
print(sber.shape)

---
## Task 3 — Loan applicants with gaps

**The story.** A bank's online form lets people skip fields. Before scoring, an analyst must know **where the gaps are**, and then fill or drop them on purpose, not by accident.

In [ ]:
# Data for Task 3: loan applicants. Income is in thousand RUB a month; NaN / None = the field was left empty.
clients = pd.DataFrame({
    "client_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "age":       [25, np.nan, 41, 33, np.nan, 52, 29, 38],
    "income":    [80, 120, np.nan, 95, np.nan, 64, 70, 150],
    "region":    ["Moscow", "Kazan", "Moscow", None, "Tomsk", "Kazan", "Moscow", "Tomsk"],
})
clients

### 3.1 Discuss
1. What are `clients["income"].size` and `clients["income"].count()`?
2. How many clients have **every** field filled?
3. If you drop every row with any gap, how many clients are left? Is that what the bank wants?

In [ ]:
HAND_3 = "..."   # your hand answers

**The spec:** `missing` — a Series with the number of empty cells in each column; `n_complete` — the number of clients with no empty field. Don't change `clients`.

### 3.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 3.3 Verify *(don't edit)*

In [ ]:
assert missing.to_dict() == {"client_id": 0, "age": 2, "income": 2, "region": 1}, missing.to_dict()
assert n_complete == 4, f"only 101, 106, 107 and 108 have every field, got {n_complete}"
assert clients.shape == (8, 4) and clients["income"].isnull().sum() == 2, "don't change `clients` itself"
print("OK:", int(missing.sum()), "empty cells;", n_complete, "complete clients of", len(clients))

**Read the code.**
1. In `clients.isnull().sum()`, what is being added up, and in which direction (down the columns or along the rows)?
2. `isnull().sum().sum()` is 5, but only 4 clients have a gap. Why are the two numbers different?

### 3.4 Feature — clean on purpose
**The spec:** `clients_clean` — drop only the clients with **no age**, then fill the empty incomes with the **median income of all 8 clients** (`clients["income"].median()`). Keep the empty region of client 104.

*Before you run:* how many rows will `clients_clean` have? What income will client 103 get?

### 3.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert len(clients_clean) == 6, f"only clients without an age are dropped: expected 6 rows, got {len(clients_clean)}"
assert list(clients_clean["client_id"]) == [101, 103, 104, 106, 107, 108], list(clients_clean["client_id"])
assert clients_clean["income"].notnull().all(), "every income must be filled"
assert clients_clean.loc[clients_clean["client_id"] == 103, "income"].iloc[0] == 87.5, \
    "103 gets the median of all 8 clients' known incomes (not the mean, not the median after dropping)"
assert clients_clean["region"].isnull().sum() == 1, "client 104 keeps its empty region: we were not asked to drop it"
assert clients["income"].isnull().sum() == 2, "don't change `clients` itself"
print("OK:", len(clients_clean), "clients kept; empty incomes filled with", clients["income"].median())

---
# B · Select
Masks, `loc` / `iloc`, and `shift`: pick the rows you need, and compare a day with the day before.

---
## Task 4 — Loud days on SBER

**The story.** Traders call a day **loud** when far more shares change hands than usual. Here, "usual" is the median daily volume, and a loud day has more than twice that. Also: how often did SBER close above its open?

### 4.1 Discuss
1. Guess the share of days with CLOSE > OPEN: closer to 30 %, 50 % or 70 %?
2. What does `(sber["CLOSE"] > sber["OPEN"]).mean()` return, and why is it a share?
3. Why the **median** volume and not the mean, as the "usual" level?

In [ ]:
HAND_4 = "..."   # your hand answers

**The spec:** `share_up` — the share of days with CLOSE > OPEN (a number between 0 and 1); `n_loud` — the number of days with VOLUME > 2 × the median VOLUME.

### 4.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 4.3 Verify *(don't edit)*

In [ ]:
assert 0 < share_up < 1, f"a share between 0 and 1, not a count: {share_up}"
assert np.isclose(share_up, 123 / 254), share_up
assert n_loud == 26, f"VOLUME > 2 x MEDIAN volume: expected 26 days, got {n_loud}"
print(f"OK: SBER closed above its open on {share_up:.1%} of days; {n_loud} loud days")

**Read the code.**
1. Which line makes a True/False column? Which turns it into a share, and which into a count?
2. With the **mean** instead of the median, would there be more loud days or fewer?

### 4.4 Feature — loud **and** red
**The spec:** `loud_red` — the rows of `sber` that are loud **and** closed below the open; `top3` — the 3 of them with the largest VOLUME, largest first. Use **`sort_values`** (new: it sorts the rows by a column).

*Before you run:* in which month do you expect the loudest red days?

### 4.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert len(loud_red) == 12, f"loud AND down: expected 12 days, got {len(loud_red)}"
assert (loud_red["CLOSE"] < loud_red["OPEN"]).all(), "every row must be a down day"
assert len(top3) == 3 and top3["VOLUME"].is_monotonic_decreasing, "3 rows, largest volume first"
assert [d.strftime("%Y-%m-%d") for d in top3.index] == ["2025-02-14", "2025-02-13", "2025-06-06"], list(top3.index)
print("OK: the loudest red day was", top3.index[0].date())

---
## Task 5 — Dates vs positions

**The story.** `loc` picks rows by **label** (here, dates); `iloc` by **position** (0, 1, 2, …). Today you need December, the first week of the file, and one closing price.

### 5.1 Discuss
1. How many trading days were there in December 2025? Count them on a calendar: the weekdays from 1 to 30 December (31 December was a holiday on MOEX).
2. How many rows do `sber.iloc[0:5]` and `sber.loc["2025-12-22":"2025-12-26"]` return? Why is the second one not 4?

In [ ]:
HAND_5 = "..."   # your hand answers

**The spec:** `dec` — all the December rows (use `.loc` with a date string); `first_5` — the first 5 rows, by position; `close_mar_31` — the CLOSE on 31 March 2025, as a **single number**.

### 5.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 5.3 Verify *(don't edit)*

In [ ]:
assert len(dec) == 22 and (dec.index.month == 12).all(), f"December 2025 has 22 trading days, got {len(dec)}"
assert len(first_5) == 5 and first_5.index[0] == pd.Timestamp("2025-01-03"), "the first five rows of the file"
assert first_5.index[-1] == pd.Timestamp("2025-01-10"), first_5.index[-1]
assert np.ndim(close_mar_31) == 0, "one number, not a table"
assert np.isclose(close_mar_31, 309.72), close_mar_31
print("OK: December has", len(dec), "days; the close on 31 Mar was", close_mar_31)

**Read the code.**
1. `sber.loc["2025-03-31", "CLOSE"]` and `sber.loc[["2025-03-31"], "CLOSE"]`: one gives a number, the other a Series. Which is which?
2. Which dates are in `first_5`? Why isn't 7 January there?

### 5.4 Feature — one week, two columns
**The spec:** `week` — the CLOSE and VOLUME columns for 22–26 December 2025, selected with `.loc` by dates; `last_close` — the CLOSE of the **last** row of the file, with `.iloc`.

*Before you run:* what is the shape of `week`?

### 5.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert week.shape == (5, 2), f"loc includes the end date: 22-26 Dec is 5 rows x 2 columns, got {week.shape}"
assert list(week.columns) == ["CLOSE", "VOLUME"], list(week.columns)
assert week.index[-1] == pd.Timestamp("2025-12-26")
assert np.ndim(last_close) == 0 and np.isclose(last_close, 299.9), last_close
print("OK: the week of 22 Dec has", len(week), "days; the year closed at", last_close)

---
## Task 6 — Returns the right way

**The story.** In Lecture 4 you computed daily returns with NumPy as `close[1:] / close[:-1] - 1`. In Lecture 6 you saw that the same idea in pandas gives zeros, because pandas divides **by label**, not by position. Ask the model for daily returns the pandas way.

### 6.1 Discuss
1. By hand: the return on 6 January 2025 (close 274.37; on 3 January, 272.25).
2. 254 prices give how many returns? Where does pandas put the missing one?

In [ ]:
HAND_6 = "..."   # your hand answers

**The spec:** `ret` — the daily returns of CLOSE, on the same 254 dates as `sber` (the first one is `NaN`); `best_day` — the date of the largest return.

### 6.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 6.3 Verify *(don't edit)*

In [ ]:
assert len(ret) == 254 and ret.index.equals(sber.index), "one value per day, on the same dates"
assert np.isnan(ret.iloc[0]) and ret.count() == 253, "the first day has no previous close: NaN, then 253 returns"
assert np.isclose(ret.loc["2025-01-06"], 274.37 / 272.25 - 1), "today / yesterday - 1, not yesterday / today"
assert pd.Timestamp(best_day) == pd.Timestamp("2025-02-12"), best_day
print(f"OK: 253 returns; best day {pd.Timestamp(best_day).date()}, {ret.max():+.2%}")

**Read the code.**
1. What does `shift(1)` do to the CLOSE column? On which row does 272.25 end up?
2. Why does `close.iloc[1:] / close.iloc[:-1]` give 1.0 almost everywhere?

### 6.4 Feature — the overnight gap
The **gap** is how far the market opens from yesterday's close: `OPEN today / CLOSE yesterday − 1`.

**The spec:** `gap` — a Series on the same dates as `sber` (first value `NaN`); `n_gap_up` — the number of days with a gap above +1 %.

*Before you run:* the gap on 6 January (open 270.88)?

### 6.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert len(gap) == 254 and np.isnan(gap.iloc[0]), "no gap on the first day: there is no yesterday"
assert np.isclose(gap.loc["2025-01-06"], 270.88 / 272.25 - 1), "today's OPEN / yesterday's CLOSE - 1"
assert n_gap_up == 3, f"expected 3 days with a gap above +1 %, got {n_gap_up}"
print("OK:", n_gap_up, "gaps above +1 %; the largest on", gap.idxmax().date(), f"{gap.max():+.2%}")

---
# C · Combine
Two tables become one: add by label, stack with `concat`, look up with `merge`.

---
## Task 7 — Two brokerage accounts

**The story.** You have two brokerage accounts. Each is a `Series`: ticker → number of shares. Some tickers are in both, some in only one. How much of each do you hold **in total**, and what is it worth?

In [ ]:
# Data for Task 7: shares held in two brokerage accounts, and today's prices (rounded for the exercise)
a = pd.Series({"SBER": 100, "GAZP": 300, "LKOH": 5})
b = pd.Series({"SBER": 50, "YDEX": 10, "MGNT": 4})
prices = pd.Series({"SBER": 300.0, "GAZP": 120.0, "LKOH": 6000.0, "YDEX": 4000.0, "MGNT": 3500.0, "VTBR": 70.0})

### 7.1 Discuss
1. By hand: the total shares of each ticker.
2. What will `a + b` show for GAZP, and why?

In [ ]:
HAND_7 = "..."   # your hand answers

**The spec:** `total` — a Series with the total number of shares of each ticker over both accounts: one value per ticker, no `NaN`.

### 7.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 7.3 Verify *(don't edit)*

In [ ]:
assert total.notnull().all(), "no NaN: a ticker missing from one account means 0 shares there"
assert set(total.index) == {"SBER", "GAZP", "LKOH", "YDEX", "MGNT"}, list(total.index)
assert total["SBER"] == 150 and total["GAZP"] == 300 and total["MGNT"] == 4, total.to_dict()
print("OK:", total.to_dict())

**Read the code.**
1. Why is `total` a float Series, although every number of shares is a whole number?
2. In what order are the tickers in `total`? Who decided it?

### 7.4 Feature — what is it worth?
`prices` has a price for every ticker you hold, and one you don't (VTBR).

**The spec:** `position_value` — total shares × price, one value per ticker **you hold** (5 values, no `NaN`); `total_value` — their sum.

*Before you run:* what happens to VTBR in `total * prices`?

### 7.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert len(position_value) == 5 and position_value.notnull().all(), "one value per ticker you hold, no NaN"
assert "VTBR" not in position_value.index, "you hold no VTBR"
assert np.isclose(position_value["SBER"], 45_000), "SBER: 150 shares x 300"
assert np.isclose(total_value, 165_000), total_value
print("OK: both accounts together are worth", f"{total_value:,.0f}", "RUB")

---
## Task 8 — Two quarters of trades

**The story.** Your broker exports trades one quarter at a time. You want **one** table for the second half of 2025.

In [ ]:
# Data for Task 8: your trades, one table per quarter. qty = number of shares, price = RUB per share.
q3 = pd.DataFrame({
    "date":   ["2025-07-03", "2025-08-12", "2025-09-05", "2025-09-25"],
    "ticker": ["SBER", "GAZP", "LKOH", "SBER"],
    "side":   ["buy", "buy", "buy", "sell"],
    "qty":    [10, 50, 1, 5],
    "price":  [310.0, 130.0, 6500.0, 295.0],
})
q4 = pd.DataFrame({
    "date":   ["2025-10-08", "2025-10-21", "2025-11-14", "2025-12-02", "2025-12-19"],
    "ticker": ["YDEX", "SBER", "GAZP", "MGNT", "VTBR"],
    "side":   ["buy", "buy", "sell", "buy", "buy"],
    "qty":    [2, 10, 20, 3, 100],
    "price":  [4100.0, 290.0, 120.0, 3300.0, 70.0],
})

### 8.1 Discuss
1. How many rows will the combined table have?
2. `q3` has row labels 0–3 and `q4` 0–4. What labels will the combined table have if you just stack them? Why does that matter for `.loc`?

In [ ]:
HAND_8 = "..."   # your hand answers

**The spec:** `trades` — the Q3 rows, then the Q4 rows, in one DataFrame with the row labels 0…8.

### 8.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 8.3 Verify *(don't edit)*

In [ ]:
assert len(trades) == 9, f"4 + 5 = 9 trades, got {len(trades)}"
assert list(trades.index) == list(range(9)), f"row labels must be 0..8 with no repeats: {list(trades.index)}"
assert {"date", "ticker", "side", "qty", "price"} <= set(trades.columns), list(trades.columns)
assert trades["ticker"].iloc[0] == "SBER" and trades["ticker"].iloc[-1] == "VTBR", "Q3 first, then Q4"
print("OK:", len(trades), "trades in one table")

**Read the code.**
1. What does `pd.concat([q3, q4], axis=1)` do instead? What shape would it have?
2. Without `ignore_index=True`, what does `trades.loc[0]` return?

### 8.4 Feature — label the quarter, then filter
**The spec:** `trades` gets a column `quarter` ("Q3" or "Q4"); `watch` — only the trades in SBER or GAZP. Use **`isin`** (new: "is the value one of these?").

*Before you run:* how many rows will `watch` have?

### 8.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert trades["quarter"].value_counts().to_dict() == {"Q4": 5, "Q3": 4}, trades["quarter"].value_counts().to_dict()
assert trades.loc[4, "quarter"] == "Q4", "row 4 is the first Q4 trade"
assert len(watch) == 5 and set(watch["ticker"]) == {"SBER", "GAZP"}, list(watch["ticker"])
print("OK:", len(watch), "trades in SBER or GAZP")

---
## Task 9 — Add sector and name to the trades

**The story.** The trades table has only tickers. A reference table `tickers` has the company name and sector, **one row per ticker**. Join them, and don't lose any trade: one ticker (VTBR) is missing from the reference table.

In [ ]:
# Data for Task 9: the 9 trades from Task 8, and a reference table with one row per ticker
trades = pd.DataFrame({
    "date":   ["2025-07-03", "2025-08-12", "2025-09-05", "2025-09-25", "2025-10-08",
               "2025-10-21", "2025-11-14", "2025-12-02", "2025-12-19"],
    "ticker": ["SBER", "GAZP", "LKOH", "SBER", "YDEX", "SBER", "GAZP", "MGNT", "VTBR"],
    "side":   ["buy", "buy", "buy", "sell", "buy", "buy", "sell", "buy", "buy"],
    "qty":    [10, 50, 1, 5, 2, 10, 20, 3, 100],
    "price":  [310.0, 130.0, 6500.0, 295.0, 4100.0, 290.0, 120.0, 3300.0, 70.0],
})
tickers = pd.DataFrame({
    "ticker": ["SBER", "GAZP", "LKOH", "YDEX", "MGNT"],
    "name":   ["Sberbank", "Gazprom", "Lukoil", "Yandex", "Magnit"],
    "sector": ["Banks", "Oil & gas", "Oil & gas", "IT", "Retail"],
})

### 9.1 Discuss
1. Which ticker in `trades` is not in `tickers`?
2. How many rows after an **inner** merge? After a **left** merge?
3. Each ticker appears once in `tickers` but several times in `trades`. Which `validate=` value describes that?

In [ ]:
HAND_9 = "..."   # your hand answers

**The spec:** `full` — `trades` with `name` and `sector` added from `tickers`; all 9 trades kept, in the same order (a left merge, with `validate="many_to_one"`).

### 9.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 9.3 Verify *(don't edit)*

In [ ]:
assert len(full) == len(trades) == 9, f"every trade must stay: expected 9 rows, got {len(full)}"
assert {"name", "sector"} <= set(full.columns), list(full.columns)
assert full["sector"].isnull().sum() == 1, "VTBR is not in the reference table, so its sector is NaN"
assert list(full["ticker"]) == list(trades["ticker"]), "the same trades, in the same order"
assert full.loc[full["ticker"] == "LKOH", "sector"].iloc[0] == "Oil & gas"
print("OK:", len(full), "trades,", full["sector"].notnull().sum(), "with a sector")

**Read the code.**
1. What would `validate="many_to_one"` do if `tickers` had SBER twice?
2. Which `how=` does `merge` use if you don't say?

### 9.4 Feature — value, and the trades to fix
**The spec:** add `value` = qty × price to `full`; `unknown` — the trades whose sector is missing.

*Before you run:* the value of the first trade? Which rows go into `unknown`?

### 9.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert full.loc[0, "value"] == 3100, "10 shares x 310"
assert full["value"].sum() == 47_975, full["value"].sum()
assert len(unknown) == 1 and unknown["ticker"].iloc[0] == "VTBR", list(unknown["ticker"])
print("OK: total traded value", f"{full['value'].sum():,.0f}", "RUB; unknown ticker:", unknown["ticker"].iloc[0])

---
# D · Summarise
One row per group: `groupby`, `resample` and `pivot_table`.

---
## Task 10 — Turnover by sector

**The story.** The risk team asks: in which sectors did you trade the most? Turnover counts **both** buys and sells.

In [ ]:
# Data for Task 10: the same 9 trades, now with a sector (VTBR is a bank) and value = qty x price, RUB
deals = pd.DataFrame({
    "ticker": ["SBER", "GAZP", "LKOH", "SBER", "YDEX", "SBER", "GAZP", "MGNT", "VTBR"],
    "sector": ["Banks", "Oil & gas", "Oil & gas", "Banks", "IT", "Banks", "Oil & gas", "Retail", "Banks"],
    "side":   ["buy", "buy", "buy", "sell", "buy", "buy", "sell", "buy", "buy"],
    "value":  [3100.0, 6500.0, 6500.0, 1475.0, 8200.0, 2900.0, 2400.0, 9900.0, 7000.0],
})

### 10.1 Discuss
1. By hand: the turnover of the Banks sector.
2. How many rows will the result have?

In [ ]:
HAND_10 = "..."   # your hand answers

**The spec:** `by_sector` — a Series with the total `value` of each sector, **largest first**. Use **`sort_values`**.

### 10.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 10.3 Verify *(don't edit)*

In [ ]:
assert len(by_sector) == 4, f"one row per sector, got {len(by_sector)}"
assert list(by_sector.index) == ["Oil & gas", "Banks", "Retail", "IT"], f"largest first: {list(by_sector.index)}"
assert by_sector["Banks"] == 14_475, "3100 + 1475 + 2900 + 7000: buys and sells both count as turnover"
assert by_sector.sum() == deals["value"].sum(), "nothing lost, nothing counted twice"
print("OK:", by_sector.to_dict())

**Read the code.**
1. Without `sort_values`, in what order do the sectors come out? Why?
2. What would `deals.groupby("sector").sum()` do with the `ticker` and `side` columns?

### 10.4 Feature — a sector report card
**The spec:** `sector_card` — one row per sector, with the columns `n_trades` (how many trades), `total` (sum of value) and `avg` (mean value of one trade), sorted by `total`, largest first. Use `.agg(name=(column, function))`, as in the Lecture 6 monthly report card.

### 10.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert sector_card.shape == (4, 3), f"4 sectors x 3 columns, got {sector_card.shape}"
assert list(sector_card.columns) == ["n_trades", "total", "avg"], list(sector_card.columns)
assert sector_card.loc["Banks", "n_trades"] == 4 and sector_card["n_trades"].sum() == 9
assert np.isclose(sector_card.loc["Banks", "avg"], 14_475 / 4), "avg = mean value of ONE trade in the sector"
assert sector_card.index[0] == "Oil & gas", "sorted by total, largest first"
print("OK:")
print(sector_card)

---
## Task 11 — SBER by quarter

**The story.** A quarterly report for SBER 2025: how much was traded in each quarter, and what each quarter returned.

### 11.1 Discuss
1. How many rows will a quarterly table have?
2. Which quarter do you expect had the most turnover? (Remember February.)
3. `groupby` by month would give how many rows?

In [ ]:
HAND_11 = "..."   # your hand answers

**The spec:** `q_value` — the total turnover (`VALUE`) of each quarter, in **billion** rubles: 4 rows, in date order. Use `resample` (or `to_period("Q")` and `groupby`).

### 11.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 11.3 Verify *(don't edit)*

In [ ]:
assert len(q_value) == 4, f"one row per quarter, got {len(q_value)}"
assert np.isclose(q_value.sum(), sber["VALUE"].sum() / 1e9), "in billion RUB, and nothing lost"
assert np.isclose(q_value.iloc[0], 1013.369869), "Q1 turnover, billion RUB"
assert q_value.iloc[0] == q_value.max(), "Q1 was the busiest quarter"
print("OK: turnover by quarter, bn RUB:", q_value.round(1).tolist())

**Read the code.**
1. What are the index labels of `q_value`? Why are they the last days of quarters, not the first?
2. What would change if you wrote `"ME"` instead of `"QE"`?

### 11.4 Feature — the return of each quarter
**The spec:** `q_return` — for each quarter, the last CLOSE of the quarter / the first OPEN of the quarter − 1; `best_q` — the label of the best quarter.

*Before you run:* Q1 opened at 280.00 (3 Jan) and closed at 309.72 (31 Mar). What is its return?

### 11.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert len(q_return) == 4, f"one return per quarter, got {len(q_return)}"
assert np.isclose(q_return.iloc[0], 309.72 / 280 - 1), "Q1: last CLOSE (31 Mar) / first OPEN (3 Jan) - 1"
assert np.allclose(q_return.values, [0.106143, 0.018196, -0.089457, 0.038147], atol=1e-6), q_return.values
assert best_q == q_return.idxmax() and best_q == q_return.index[0], "best_q is the label of the best quarter: Q1"
print("OK: quarterly returns", [f"{r:+.2%}" for r in q_return])

---
## Task 12 — Budget by month and category

**The story.** Your banking app exports every payment as a row. You want the classic budget view: **months down, categories across**.

In [ ]:
# Data for Task 12: your spending, one row per payment, RUB
expenses = pd.DataFrame({
    "month":    ["2025-09"] * 5 + ["2025-10"] * 3 + ["2025-11"] * 4,
    "category": ["Rent", "Food", "Food", "Transport", "Fun",
                 "Rent", "Food", "Transport",
                 "Rent", "Food", "Transport", "Fun"],
    "amount":   [40_000, 12_000, 8_000, 3_000, 5_000,
                 40_000, 18_500, 3_500,
                 42_000, 17_000, 2_500, 9_000],
})
expenses

### 12.1 Discuss
1. By hand: Food in September.
2. What shape will the table have? Which cell has no payments at all?
3. Should that cell be `NaN` or 0?

In [ ]:
HAND_12 = "..."   # your hand answers

**The spec:** `budget` — one row per month, one column per category, each cell the **total** amount, empty cells = 0. Use **`pivot_table`** (new: `groupby` plus a reshape, in one call).

### 12.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 12.3 Verify *(don't edit)*

In [ ]:
assert len(budget) == 3, f"one row per month, got {len(budget)}"
assert {"Food", "Fun", "Rent", "Transport"} <= set(budget.columns), list(budget.columns)
assert budget.loc["2025-09", "Food"] == 20_000, "two Food payments in September: 12 000 + 8 000 (sum, not mean)"
assert budget.loc["2025-10", "Fun"] == 0 and budget.notnull().all().all(), "no Fun in October: 0, not NaN"
assert budget[["Food", "Fun", "Rent", "Transport"]].values.sum() == expenses["amount"].sum(), "nothing lost"
print("OK:")
print(budget)

**Read the code.**
1. What is the default `aggfunc` of `pivot_table`? What would September Food be with it?
2. How would you get the same table with `groupby(["month", "category"])` and one more step?

### 12.4 Feature — totals and the rent share
**The spec:** add a column `Total` (the sum of the four categories in each month); `rent_share` — Rent / Total for each month.

*Before you run:* which month has the highest rent share, and why?

### 12.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert budget["Total"].tolist() == [68_000, 62_000, 70_500], budget["Total"].tolist()
assert len(rent_share) == 3 and np.isclose(rent_share["2025-09"], 40_000 / 68_000), rent_share
assert rent_share.idxmax() == "2025-10", "October: the same rent, but less spent on everything else"
print("OK: rent takes", [f"{x:.1%}" for x in rent_share], "of the budget")

---
## AI log

Required by HSE AI-disclosure rules, and by this course regardless.

| Task | Model used | Prompts until the check passed | What did you have to tell it the second time? |
|---|---|---|---|
| 1 Portfolio | | | |
| 1 Feature | | | |
| 2 Load SBER | | | |
| 2 Feature | | | |
| 3 Gaps | | | |
| 3 Feature | | | |
| 4 Loud days | | | |
| 4 Feature | | | |
| 5 loc / iloc | | | |
| 5 Feature | | | |
| 6 Returns | | | |
| 6 Feature | | | |
| 7 Two accounts | | | |
| 7 Feature | | | |
| 8 concat | | | |
| 8 Feature | | | |
| 9 merge | | | |
| 9 Feature | | | |
| 10 groupby | | | |
| 10 Feature | | | |
| 11 Quarters | | | |
| 11 Feature | | | |
| 12 Budget | | | |
| 12 Feature | | | |

**One sentence:** which check caught something you would not have noticed by reading the output?

*(answer here)*